# Siamese matcher tuning

This Kaggle notebook trains and evaluates the repository's timm Siamese scene-pair classifier. It is a separate same-area classification task, not a replacement for SIFT, ORB, LoFTR, or LightGlue local correspondences.

## Configuration

Use the geographic split by default. Enable `ALLOW_PAIR_LEVEL_FALLBACK` only as a technical smoke test; its scores are not valid geographic generalization metrics.

In [20]:
from pathlib import Path
import re
import shutil

INPUT_ROOT = Path('/kaggle/input')
PROJECT_ROOT = Path('/kaggle/working')
CODE_ROOT = PROJECT_ROOT / 'src' / 'sentinel_matching'
OUTPUT_ROOT = PROJECT_ROOT / 'siamese_matcher_output'
MANIFEST = OUTPUT_ROOT / 'pairs.csv'
CHECKPOINT = OUTPUT_ROOT / 'siamese_bce.pt'
BANDS = 'B04,B03,B02'
BACKBONE = 'efficientnet_b0'
OBJECTIVE = 'bce'
EPOCHS = 10
BATCH_SIZE = 8
IMAGE_SIZE = 224
LEARNING_RATE = 2e-4
WORKERS = 2
DEVICE = 'cuda'
ALLOW_PAIR_LEVEL_FALLBACK = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
archives = sorted({path.parent for path in INPUT_ROOT.rglob('train_matcher.py') if path.parent.name == 'sentinel_matching'}, key=str)
SOURCE_ROOT = archives[0] if archives else None
if SOURCE_ROOT is None:
    print('code_found=False; upload the sentinel_matching archive as a Kaggle dataset.')
else:
    shutil.copytree(SOURCE_ROOT, CODE_ROOT, dirs_exist_ok=True)
    print(f'code_found=True\ncode_root={CODE_ROOT}')

code_found=True
code_root=/kaggle/working/src/sentinel_matching


## Dependencies

The notebook retains Kaggle's CUDA PyTorch build and installs the project packages needed for the timm training script.

In [21]:
if SOURCE_ROOT is not None:
    !python -m pip install -q timm==1.0.15 scikit-learn==1.6.1 rasterio==1.4.3 click==8.1.8 tqdm==4.67.1
    import timm
    import torch
    print(f'timm={timm.__version__}\ntorch={torch.__version__}\ncuda_available={torch.cuda.is_available()}')
else:
    print('dependency_installation_skipped=True')

timm=1.0.15
torch=2.10.0+cu128
cuda_available=True


## Geographic split check

A valid split needs at least three MGRS tile groups, each containing at least two acquisition dates. This prevents geographic texture from leaking between training, validation, and test.

In [22]:
def find_safe_scenes(root):
    return sorted((path.resolve() for path in root.rglob('*.SAFE') if path.is_dir()), key=str)

def tile_id(scene):
    match = re.search(r'_T(\d{2}[A-Z]{3})_', scene.name.upper())
    return match.group(1) if match else 'unknown'


SAFE_SCENES = find_safe_scenes(INPUT_ROOT)
GROUPS = {}
for scene in SAFE_SCENES:
    GROUPS.setdefault(tile_id(scene), []).append(scene)
ELIGIBLE_GROUPS = {name: scenes for name, scenes in GROUPS.items() if len(scenes) >= 2}
GEOGRAPHIC_SPLIT_READY = len(ELIGIBLE_GROUPS) >= 3
TRAINING_READY = SOURCE_ROOT is not None and (GEOGRAPHIC_SPLIT_READY or ALLOW_PAIR_LEVEL_FALLBACK)
print(f'safe_scenes={len(SAFE_SCENES)}')
print(f'eligible_tile_groups={len(ELIGIBLE_GROUPS)}')
for name, scenes in ELIGIBLE_GROUPS.items():
    print(f'{name}: {len(scenes)} scenes')
print(f'geographic_split_ready={GEOGRAPHIC_SPLIT_READY}')
print(f'training_ready={TRAINING_READY}')

safe_scenes=50
eligible_tile_groups=2
36UYA: 38 scenes
36UXA: 12 scenes
geographic_split_ready=False
training_ready=True


## Create the pair manifest

The manifest contains same-tile positives and different-tile negatives. The normal path preserves disjoint geographic train, validation, and test groups.

In [16]:
!python src/sentinel_matching/pair_manifest.py --root "{INPUT_ROOT}" --output "{MANIFEST}" --max-positive-pairs-per-tile 20 {FALLBACK_FLAG}

Scanning SAFE scenes: 100%|█████████████████| 50/50 [00:00<00:00, 211619.78it/s]
saved=/kaggle/working/siamese_matcher_output/pairs.csv pairs=80 positives=40 negatives=40


## Tune the timm Siamese matcher

The model uses a pretrained EfficientNet-B0 encoder and BCE loss. Each epoch prints training loss and validation metrics; the checkpoint with the best validation F1 is retained.

In [23]:
if MANIFEST.is_file():
    !python src/sentinel_matching/train_matcher.py --manifest "{MANIFEST}" --output "{CHECKPOINT}" --backbone "{BACKBONE}" --objective "{OBJECTIVE}" --bands "{BANDS}" --image-size {IMAGE_SIZE} --batch-size {BATCH_SIZE} --epochs {EPOCHS} --learning-rate {LEARNING_RATE} --workers {WORKERS} --device "{DEVICE}" --disable-wandb
else:
    print('training_skipped=True; manifest is unavailable.')

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
{"training_device": "cuda", "backbone": "efficientnet_b0", "train_pairs": 56, "validation_pairs": 12, "batch_size": 8, "epochs": 10}
Epoch 1/10 train:   0%|                                   | 0/7 [03:38<?, ?it/s]

Aborted!


## Held-out evaluation

The script loads the validation-selected checkpoint and evaluates it once on the held-out test split, printing accuracy, precision, recall, F1, and loss.

In [24]:
if CHECKPOINT.is_file() and MANIFEST.is_file():
    !python src/sentinel_matching/evaluate_matcher.py --checkpoint "{CHECKPOINT}" --manifest "{MANIFEST}" --batch-size {BATCH_SIZE} --workers {WORKERS} --device "{DEVICE}" --disable-wandb
    print(f'checkpoint={CHECKPOINT}')
else:
    print('evaluation_skipped=True; no trained checkpoint was created.')

evaluation_skipped=True; no trained checkpoint was created.
